# 07 Results Freeze

Final reviewer evidence matrix and artifact freeze.

## Setup

Clone/pull latest code into `/content/ECG-RAMBA`, restore verified Drive mirror artifacts, and define a logging command helper. Notebook 07 is self-contained: you do not need to run Notebook 00 first. If you skip this setup cell and start from **Required Input Check**, that cell contains a direct-run guard that performs the same lightweight setup before freezing final evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

try:
    from google.colab import drive
    if not _drive_root_ready(DRIVE_ROOT):
        try:
            drive.mount(str(DRIVE_MOUNT))
        except Exception as exc:
            print(f'Drive mount initial attempt failed or was stale: {exc}')
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
    else:
        print('Drive root already visible:', DRIVE_ROOT)
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path('/content/ECG-RAMBA')

os.chdir('/content')
if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout. Rename it or use a fresh runtime.')
if not REPO_DIR.exists():
    print(f'$ git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
    subprocess.run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', shell=True, check=True)

os.chdir(REPO_DIR)

def run(cmd, check=True, log_path=None, tail_lines=120, cwd=None):
    run_cwd = Path(cwd) if cwd else Path.cwd()
    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f'notebook_command_{time.strftime("%Y%m%d_%H%M%S")}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
    except ValueError:
        log_relative = Path('logs') / log_path.name
    durable_log_path = MIRROR_REVISION_ROOT / log_relative
    durable_log_path.parent.mkdir(parents=True, exist_ok=True)

    from contextlib import ExitStack
    with ExitStack() as stack:
        log = stack.enter_context(log_path.open('a', encoding='utf-8'))
        durable_log = stack.enter_context(durable_log_path.open('a', encoding='utf-8'))
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            durable_log.write(line)
            durable_log.flush()
        rc = proc.wait()
    print(f'Command log: {log_path}')
    print(f'Durable command log: {durable_log_path}')
    if check and rc:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(rc, cmd)
    return subprocess.CompletedProcess(cmd, rc)

run('git fetch origin', check=False)
run(f'git checkout {BRANCH}', check=True)
run(f'git pull --ff-only --autostash origin {BRANCH}', check=True)

# BEGIN FORENSIC CODE AUTHORITY PIN
FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_versioned_git_release_v2'
FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 2
_AUTHORITY_BOOTSTRAP_ALLOWED = False
_AUTHORITY_DEFAULT_RELEASE_REF = 'refs/tags/ecg-ramba-revision-20260723-v11'

def _pin_forensic_code_authority():
    import json as _authority_json
    import os as _authority_os
    import re as _authority_re
    import subprocess as _authority_subprocess
    import uuid as _authority_uuid
    from datetime import datetime as _authority_datetime, timezone as _authority_timezone
    from pathlib import Path as _AuthorityPath
    from scripts.revision.artifact_mirror import PublishLock as _AuthorityPublishLock

    manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
    requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
    requested_release_ref = _authority_os.environ.get(
        'ECG_RAMBA_AUTHORITY_REF', _AUTHORITY_DEFAULT_RELEASE_REF
    ).strip()
    reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
    legacy_rotate_requested = (
        _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '0') == '1'
    )
    if legacy_rotate_requested:
        raise RuntimeError(
            'Implicit authority rotation to a moving branch head is disabled. '
            'Notebook 00 follows only a reviewed versioned release tag. For an emergency explicit '
            'pin, set ECG_RAMBA_RESET_CODE_AUTHORITY=1 with a full ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    rotate_to_branch_head = False
    commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')
    release_ref_pattern = _authority_re.compile(r'refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*')

    def git(*args, check=True):
        result = _authority_subprocess.run(
            ['git', *args],
            cwd=str(REPO_DIR),
            check=False,
            text=True,
            stdout=_authority_subprocess.PIPE,
            stderr=_authority_subprocess.STDOUT,
        )
        if check and result.returncode:
            raise RuntimeError(
                'Code-authority git command failed: git '
                + ' '.join(args)
                + '\n'
                + (result.stdout or '')[-4000:]
            )
        return result

    def resolve_annotated_release_ref(release_ref):
        if not release_ref_pattern.fullmatch(release_ref):
            raise RuntimeError(
                'ECG_RAMBA_AUTHORITY_REF must be a full refs/tags/... name. '
                'Moving branches are not accepted as code authority.'
            )
        object_type = git('cat-file', '-t', release_ref, check=False)
        if object_type.returncode or object_type.stdout.strip() != 'tag':
            raise RuntimeError(
                f'Code-authority release ref is missing or is not an annotated Git tag: {release_ref}. '
                'Run the current Notebook 00 after the reviewed release tag has been pushed.'
            )
        object_id = git('rev-parse', '--verify', release_ref).stdout.strip().lower()
        commit = git('rev-parse', '--verify', release_ref + '^{commit}').stdout.strip().lower()
        if not commit_pattern.fullmatch(object_id) or not commit_pattern.fullmatch(commit):
            raise RuntimeError(f'Could not resolve an immutable object and commit for {release_ref}.')
        return commit, object_id

    if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
        raise RuntimeError(
            'Only Notebook 00 may rotate the canonical code authority. '
            'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
            'ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    if reset_requested and not commit_pattern.fullmatch(requested_commit):
        raise RuntimeError(
            'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
        )

    tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
    if tracked_status:
        raise RuntimeError(
            'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
            'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
        )

    fetch = git('fetch', 'origin', '--prune', '--tags', check=False)
    if fetch.returncode:
        print('WARNING: git fetch failed; accepting only an already-present pinned commit/release tag.')
        print((fetch.stdout or '')[-2000:])

    manifest = None
    manifest_raw = None
    with _AuthorityPublishLock(
        _AuthorityPath(MIRROR_REVISION_ROOT),
        run_id='authority-read-' + _authority_uuid.uuid4().hex,
    ):
        if manifest_path.is_file():
            manifest_raw = manifest_path.read_text(encoding='utf-8')
            manifest = _authority_json.loads(manifest_raw)

    authority_update_needed = False
    update_reason = None
    release_ref = None
    release_ref_object_id = None

    if reset_requested:
        expected_commit = requested_commit
        authority_update_needed = True
        update_reason = 'explicit_environment_sha'
    elif manifest is None:
        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise FileNotFoundError(
                'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                'downstream notebooks fail closed instead of following a moving branch.'
            )
        release_ref = requested_release_ref
        expected_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
        authority_update_needed = True
        update_reason = 'initial_versioned_release_bootstrap'
    else:
        manifest_capability = manifest.get('capability')
        manifest_schema = int(manifest.get('schema_version', 0))
        legacy_manifest = (
            manifest_capability == 'canonical_git_commit_pin_v1' and manifest_schema == 1
        )
        current_manifest = (
            manifest_capability == 'canonical_versioned_git_release_v2' and manifest_schema == 2
        )
        if not legacy_manifest and not current_manifest:
            raise RuntimeError('Canonical code-authority manifest capability/schema is invalid.')
        manifest_commit = str(manifest.get('git_commit', '')).strip().lower()
        if not commit_pattern.fullmatch(manifest_commit):
            raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
        if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
            raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
        if str(manifest.get('branch', '')) != str(BRANCH):
            raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')

        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            if legacy_manifest:
                raise RuntimeError(
                    'Canonical code authority uses the legacy schema. Run the current Notebook 00 once '
                    'to migrate it to the reviewed versioned release before running downstream notebooks.'
                )
            expected_commit = manifest_commit
            if requested_commit and requested_commit != expected_commit:
                raise RuntimeError(
                    'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                    'Rotate authority explicitly in Notebook 00; do not override it downstream.'
                )
        else:
            release_ref = requested_release_ref
            release_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
            manifest_ref = str(manifest.get('authority_ref', '')).strip()
            manifest_ref_object = str(manifest.get('authority_ref_object_id', '')).strip().lower()
            if current_manifest and manifest_ref == release_ref:
                if manifest_commit != release_commit or manifest_ref_object != release_ref_object_id:
                    raise RuntimeError(
                        f'Code-authority release tag moved or changed after it was recorded: {release_ref}. '
                        'Publish a new versioned tag instead of retagging an existing release.'
                    )
                expected_commit = manifest_commit
            else:
                expected_commit = release_commit
                authority_update_needed = True
                update_reason = (
                    'legacy_manifest_migration'
                    if legacy_manifest
                    else 'versioned_release_upgrade'
                )

    if not commit_pattern.fullmatch(expected_commit):
        raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')
    git('cat-file', '-e', expected_commit + '^{commit}')
    git('checkout', '--detach', expected_commit)
    observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
    if observed_commit != expected_commit:
        raise RuntimeError(
            f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
        )

    if authority_update_needed:
        previous_commit = None if manifest is None else str(manifest.get('git_commit', '')).strip().lower()
        previous_ref = None if manifest is None else manifest.get('authority_ref')
        manifest = {
            'capability': 'canonical_versioned_git_release_v2',
            'schema_version': 2,
            'git_commit': expected_commit,
            'repository_url': str(REPO_URL),
            'branch': str(BRANCH),
            'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
            'established_by': '00_colab_bootstrap.ipynb',
            'selection': (
                'explicit_environment_sha'
                if release_ref is None
                else 'verified_annotated_versioned_release_tag'
            ),
            'authority_ref': release_ref,
            'authority_ref_kind': None if release_ref is None else 'annotated_git_tag',
            'authority_ref_object_id': release_ref_object_id,
            'update_reason': update_reason,
            'previous_git_commit': previous_commit,
            'previous_authority_ref': previous_ref,
        }
        with _AuthorityPublishLock(
            _AuthorityPath(MIRROR_REVISION_ROOT),
            run_id='authority-write-' + _authority_uuid.uuid4().hex,
        ):
            concurrent_raw = manifest_path.read_text(encoding='utf-8') if manifest_path.is_file() else None
            if concurrent_raw != manifest_raw and not reset_requested:
                concurrent = _authority_json.loads(concurrent_raw) if concurrent_raw else None
                if not (
                    concurrent
                    and concurrent.get('capability') == 'canonical_versioned_git_release_v2'
                    and int(concurrent.get('schema_version', 0)) == 2
                    and str(concurrent.get('git_commit', '')).lower() == expected_commit
                    and concurrent.get('authority_ref') == release_ref
                    and str(concurrent.get('authority_ref_object_id', '')).lower()
                    == str(release_ref_object_id or '').lower()
                ):
                    raise RuntimeError('A concurrent runtime established a different code authority.')
                manifest = concurrent
            else:
                manifest_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = manifest_path.with_name(
                    manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
                )
                with temporary.open('w', encoding='utf-8') as handle:
                    handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
                    handle.flush()
                    _authority_os.fsync(handle.fileno())
                _authority_os.replace(temporary, manifest_path)
        print('Established/updated canonical code authority:', manifest_path)

    _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
    _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
    globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
    globals()['CODE_AUTHORITY'] = manifest
    print('Pinned code authority:', expected_commit)
    print('Authority release  :', manifest.get('authority_ref'))
    print('Authority manifest :', manifest_path)
    return manifest

CODE_AUTHORITY = _pin_forensic_code_authority()
# END FORENSIC CODE AUTHORITY PIN
RUN_FULL_MIRROR_RESTORE_07 = os.environ.get('ECG_RAMBA_FULL_MIRROR_RESTORE_07', '0') == '1'
if MIRROR_REVISION_ROOT.exists() and RUN_FULL_MIRROR_RESTORE_07:
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_full_restore.log',
    )
elif MIRROR_REVISION_ROOT.exists():
    print('Skipping full mirror restore in Notebook 07; the next cell restores only authenticated evidence metadata.')
else:
    print('Mirror not found; continuing with repo-local reports/revision:', MIRROR_REVISION_ROOT)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
run('git rev-parse --short HEAD', check=False)
run('git status --short --branch', check=False)

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## Required Input Check

Validate the revision-plan CSV files and the final evidence inputs before building the final matrix. This cell is safe to run directly after opening Notebook 07: if Setup was skipped, it mounts Drive, clones/pulls the active repo, defines `run()`, and restores required evidence artifacts before continuing.


In [ ]:
# Notebook 07 direct-run guard: allows starting from this cell without Notebook 00 or the Setup cell.
from pathlib import Path
import json
import os
import subprocess
import sys
import time

if (
    'run' not in globals()
    or 'DRIVE_ROOT' not in globals()
    or 'MIRROR_REVISION_ROOT' not in globals()
    or not Path.cwd().as_posix().endswith('/content/ECG-RAMBA')
):
    print('Notebook 07 setup context missing; running embedded direct setup guard.')
    DRIVE_MOUNT = Path('/content/drive')
    DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'

    def _direct_drive_root_ready(root):
        try:
            return root.is_dir() and any(root.iterdir())
        except Exception:
            return False

    try:
        from google.colab import drive
        if not _direct_drive_root_ready(DRIVE_ROOT):
            try:
                drive.mount(str(DRIVE_MOUNT))
            except Exception as exc:
                print(f'Drive mount initial attempt failed or was stale: {exc}')
                drive.mount(str(DRIVE_MOUNT), force_remount=True)
    except Exception as exc:
        print(f'Drive mount skipped or unavailable: {exc}')
    if not _direct_drive_root_ready(DRIVE_ROOT):
        raise RuntimeError(f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.')

    MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
    REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
    BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
    REPO_DIR = Path('/content/ECG-RAMBA')

    os.chdir('/content')
    if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout. Rename it or use a fresh runtime.')
    if not REPO_DIR.exists():
        print(f'$ git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
        subprocess.run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', shell=True, check=True)

    os.chdir(REPO_DIR)

    def run(cmd, check=True, log_path=None, tail_lines=120, cwd=None):
        run_cwd = Path(cwd) if cwd else Path.cwd()
        print(f'$ {cmd}', flush=True)
        if log_path is None:
            log_dir = run_cwd / 'reports' / 'revision' / 'logs'
            log_dir.mkdir(parents=True, exist_ok=True)
            log_path = log_dir / f'notebook_command_{time.strftime("%Y%m%d_%H%M%S")}.log'
        else:
            log_path = Path(log_path)
            if not log_path.is_absolute():
                log_path = run_cwd / log_path
            log_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
        except ValueError:
            log_relative = Path('logs') / log_path.name
        durable_log_path = MIRROR_REVISION_ROOT / log_relative
        durable_log_path.parent.mkdir(parents=True, exist_ok=True)

        from contextlib import ExitStack
        with ExitStack() as stack:
            log = stack.enter_context(log_path.open('a', encoding='utf-8'))
            durable_log = stack.enter_context(durable_log_path.open('a', encoding='utf-8'))
            proc = subprocess.Popen(
                cmd,
                shell=True,
                cwd=str(run_cwd),
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8',
                errors='replace',
                bufsize=1,
            )
            assert proc.stdout is not None
            for line in proc.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
                durable_log.write(line)
                durable_log.flush()
            rc = proc.wait()
        print(f'Command log: {log_path}')
        print(f'Durable command log: {durable_log_path}')
        if check and rc:
            lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
            for line in lines[-tail_lines:]:
                print(line)
            raise subprocess.CalledProcessError(rc, cmd)
        return subprocess.CompletedProcess(cmd, rc)

    run('git fetch origin', check=False)
    run(f'git checkout {BRANCH}', check=True)
    run(f'git pull --ff-only --autostash origin {BRANCH}', check=True)

    # BEGIN FORENSIC CODE AUTHORITY PIN
    FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_versioned_git_release_v2'
    FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 2
    _AUTHORITY_BOOTSTRAP_ALLOWED = False
    _AUTHORITY_DEFAULT_RELEASE_REF = 'refs/tags/ecg-ramba-revision-20260723-v11'

    def _pin_forensic_code_authority():
        import json as _authority_json
        import os as _authority_os
        import re as _authority_re
        import subprocess as _authority_subprocess
        import uuid as _authority_uuid
        from datetime import datetime as _authority_datetime, timezone as _authority_timezone
        from pathlib import Path as _AuthorityPath
        from scripts.revision.artifact_mirror import PublishLock as _AuthorityPublishLock

        manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
        requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
        requested_release_ref = _authority_os.environ.get(
            'ECG_RAMBA_AUTHORITY_REF', _AUTHORITY_DEFAULT_RELEASE_REF
        ).strip()
        reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
        legacy_rotate_requested = (
            _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '0') == '1'
        )
        if legacy_rotate_requested:
            raise RuntimeError(
                'Implicit authority rotation to a moving branch head is disabled. '
                'Notebook 00 follows only a reviewed versioned release tag. For an emergency explicit '
                'pin, set ECG_RAMBA_RESET_CODE_AUTHORITY=1 with a full ECG_RAMBA_AUTHORITY_COMMIT.'
            )
        rotate_to_branch_head = False
        commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')
        release_ref_pattern = _authority_re.compile(r'refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*')

        def git(*args, check=True):
            result = _authority_subprocess.run(
                ['git', *args],
                cwd=str(REPO_DIR),
                check=False,
                text=True,
                stdout=_authority_subprocess.PIPE,
                stderr=_authority_subprocess.STDOUT,
            )
            if check and result.returncode:
                raise RuntimeError(
                    'Code-authority git command failed: git '
                    + ' '.join(args)
                    + '\n'
                    + (result.stdout or '')[-4000:]
                )
            return result

        def resolve_annotated_release_ref(release_ref):
            if not release_ref_pattern.fullmatch(release_ref):
                raise RuntimeError(
                    'ECG_RAMBA_AUTHORITY_REF must be a full refs/tags/... name. '
                    'Moving branches are not accepted as code authority.'
                )
            object_type = git('cat-file', '-t', release_ref, check=False)
            if object_type.returncode or object_type.stdout.strip() != 'tag':
                raise RuntimeError(
                    f'Code-authority release ref is missing or is not an annotated Git tag: {release_ref}. '
                    'Run the current Notebook 00 after the reviewed release tag has been pushed.'
                )
            object_id = git('rev-parse', '--verify', release_ref).stdout.strip().lower()
            commit = git('rev-parse', '--verify', release_ref + '^{commit}').stdout.strip().lower()
            if not commit_pattern.fullmatch(object_id) or not commit_pattern.fullmatch(commit):
                raise RuntimeError(f'Could not resolve an immutable object and commit for {release_ref}.')
            return commit, object_id

        if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise RuntimeError(
                'Only Notebook 00 may rotate the canonical code authority. '
                'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
                'ECG_RAMBA_AUTHORITY_COMMIT.'
            )
        if reset_requested and not commit_pattern.fullmatch(requested_commit):
            raise RuntimeError(
                'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
            )

        tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
        if tracked_status:
            raise RuntimeError(
                'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
                'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
            )

        fetch = git('fetch', 'origin', '--prune', '--tags', check=False)
        if fetch.returncode:
            print('WARNING: git fetch failed; accepting only an already-present pinned commit/release tag.')
            print((fetch.stdout or '')[-2000:])

        manifest = None
        manifest_raw = None
        with _AuthorityPublishLock(
            _AuthorityPath(MIRROR_REVISION_ROOT),
            run_id='authority-read-' + _authority_uuid.uuid4().hex,
        ):
            if manifest_path.is_file():
                manifest_raw = manifest_path.read_text(encoding='utf-8')
                manifest = _authority_json.loads(manifest_raw)

        authority_update_needed = False
        update_reason = None
        release_ref = None
        release_ref_object_id = None

        if reset_requested:
            expected_commit = requested_commit
            authority_update_needed = True
            update_reason = 'explicit_environment_sha'
        elif manifest is None:
            if not _AUTHORITY_BOOTSTRAP_ALLOWED:
                raise FileNotFoundError(
                    'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                    'downstream notebooks fail closed instead of following a moving branch.'
                )
            release_ref = requested_release_ref
            expected_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
            authority_update_needed = True
            update_reason = 'initial_versioned_release_bootstrap'
        else:
            manifest_capability = manifest.get('capability')
            manifest_schema = int(manifest.get('schema_version', 0))
            legacy_manifest = (
                manifest_capability == 'canonical_git_commit_pin_v1' and manifest_schema == 1
            )
            current_manifest = (
                manifest_capability == 'canonical_versioned_git_release_v2' and manifest_schema == 2
            )
            if not legacy_manifest and not current_manifest:
                raise RuntimeError('Canonical code-authority manifest capability/schema is invalid.')
            manifest_commit = str(manifest.get('git_commit', '')).strip().lower()
            if not commit_pattern.fullmatch(manifest_commit):
                raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
            if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
                raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
            if str(manifest.get('branch', '')) != str(BRANCH):
                raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')

            if not _AUTHORITY_BOOTSTRAP_ALLOWED:
                if legacy_manifest:
                    raise RuntimeError(
                        'Canonical code authority uses the legacy schema. Run the current Notebook 00 once '
                        'to migrate it to the reviewed versioned release before running downstream notebooks.'
                    )
                expected_commit = manifest_commit
                if requested_commit and requested_commit != expected_commit:
                    raise RuntimeError(
                        'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                        'Rotate authority explicitly in Notebook 00; do not override it downstream.'
                    )
            else:
                release_ref = requested_release_ref
                release_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
                manifest_ref = str(manifest.get('authority_ref', '')).strip()
                manifest_ref_object = str(manifest.get('authority_ref_object_id', '')).strip().lower()
                if current_manifest and manifest_ref == release_ref:
                    if manifest_commit != release_commit or manifest_ref_object != release_ref_object_id:
                        raise RuntimeError(
                            f'Code-authority release tag moved or changed after it was recorded: {release_ref}. '
                            'Publish a new versioned tag instead of retagging an existing release.'
                        )
                    expected_commit = manifest_commit
                else:
                    expected_commit = release_commit
                    authority_update_needed = True
                    update_reason = (
                        'legacy_manifest_migration'
                        if legacy_manifest
                        else 'versioned_release_upgrade'
                    )

        if not commit_pattern.fullmatch(expected_commit):
            raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')
        git('cat-file', '-e', expected_commit + '^{commit}')
        git('checkout', '--detach', expected_commit)
        observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
        if observed_commit != expected_commit:
            raise RuntimeError(
                f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
            )

        if authority_update_needed:
            previous_commit = None if manifest is None else str(manifest.get('git_commit', '')).strip().lower()
            previous_ref = None if manifest is None else manifest.get('authority_ref')
            manifest = {
                'capability': 'canonical_versioned_git_release_v2',
                'schema_version': 2,
                'git_commit': expected_commit,
                'repository_url': str(REPO_URL),
                'branch': str(BRANCH),
                'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
                'established_by': '00_colab_bootstrap.ipynb',
                'selection': (
                    'explicit_environment_sha'
                    if release_ref is None
                    else 'verified_annotated_versioned_release_tag'
                ),
                'authority_ref': release_ref,
                'authority_ref_kind': None if release_ref is None else 'annotated_git_tag',
                'authority_ref_object_id': release_ref_object_id,
                'update_reason': update_reason,
                'previous_git_commit': previous_commit,
                'previous_authority_ref': previous_ref,
            }
            with _AuthorityPublishLock(
                _AuthorityPath(MIRROR_REVISION_ROOT),
                run_id='authority-write-' + _authority_uuid.uuid4().hex,
            ):
                concurrent_raw = manifest_path.read_text(encoding='utf-8') if manifest_path.is_file() else None
                if concurrent_raw != manifest_raw and not reset_requested:
                    concurrent = _authority_json.loads(concurrent_raw) if concurrent_raw else None
                    if not (
                        concurrent
                        and concurrent.get('capability') == 'canonical_versioned_git_release_v2'
                        and int(concurrent.get('schema_version', 0)) == 2
                        and str(concurrent.get('git_commit', '')).lower() == expected_commit
                        and concurrent.get('authority_ref') == release_ref
                        and str(concurrent.get('authority_ref_object_id', '')).lower()
                        == str(release_ref_object_id or '').lower()
                    ):
                        raise RuntimeError('A concurrent runtime established a different code authority.')
                    manifest = concurrent
                else:
                    manifest_path.parent.mkdir(parents=True, exist_ok=True)
                    temporary = manifest_path.with_name(
                        manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
                    )
                    with temporary.open('w', encoding='utf-8') as handle:
                        handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
                        handle.flush()
                        _authority_os.fsync(handle.fileno())
                    _authority_os.replace(temporary, manifest_path)
            print('Established/updated canonical code authority:', manifest_path)

        _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
        _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
        globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
        globals()['CODE_AUTHORITY'] = manifest
        print('Pinned code authority:', expected_commit)
        print('Authority release  :', manifest.get('authority_ref'))
        print('Authority manifest :', manifest_path)
        return manifest

    CODE_AUTHORITY = _pin_forensic_code_authority()
    # END FORENSIC CODE AUTHORITY PIN
    print('Direct setup guard complete.')
    print('cwd       :', Path.cwd())
    print('drive_root:', DRIVE_ROOT)
    print('repo_dir  :', REPO_DIR)
    print('branch    :', BRANCH)
else:
    print('Notebook 07 setup context already present; continuing with existing repo/session.')

import pandas as pd

revision_root = Path('reports/revision')

# Restore the complete evidence metadata surface without copying multi-GB
# checkpoints, raw stress caches, or representation embeddings into /content.
# This prevents optional evidence from being falsely marked missing in a fresh runtime.
EVIDENCE_RESTORE_PREFIXES = ['metrics/', 'tables/', 'manifests/', 'figures/']
EVIDENCE_RESTORE_PATHS = [
    'a0_resolution_status.json',
    'audit_protocol.json',
    'predictions/oof_final_ema_predictions.npz',
    'predictions/transformer_ecg_oof_predictions.npz',
    'predictions/hybrid_morphology_oof_predictions.npz',
    'predictions/morphology_learnability_frozen_oof_predictions.npz',
    'predictions/morphology_learnability_partial_oof_predictions.npz',
    'experimental/external/ptbxl/ptbxl_full_predictions.npz',
    'experimental/external/ptbxl/ptbxl_full_slice_predictions.npz',
    'experimental/external/georgia/georgia_full_predictions.npz',
    'experimental/external/georgia/georgia_full_slice_predictions.npz',
    'experimental/external/cpsc2021/cpsc2021_full_predictions.npz',
    'experimental/external/cpsc2021/cpsc2021_full_slice_predictions.npz',
]
if not MIRROR_REVISION_ROOT.exists():
    raise FileNotFoundError(f'Canonical artifact mirror does not exist: {MIRROR_REVISION_ROOT}')
evidence_restore_args = ' '.join(
    [f'--include-prefix "{item}"' for item in EVIDENCE_RESTORE_PREFIXES]
    + [f'--include-path "{item}"' for item in EVIDENCE_RESTORE_PATHS]
)
run(
    f'python -u scripts/revision/artifact_mirror.py restore '
    f'--mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched '
    f'--exclude-cache-dirs {evidence_restore_args}',
    log_path='reports/revision/logs/final_evidence_targeted_restore.log',
)
plan_csvs = [
    Path('docs/revision_plan/experiment_registry.csv'),
    Path('docs/revision_plan/claim_evidence_map.csv'),
    Path('docs/revision_plan/task_board.csv'),
]
for csv_path in plan_csvs:
    df = pd.read_csv(csv_path)
    print(f'{csv_path}: rows={len(df)} cols={len(df.columns)}')

required_final_inputs = {
    'calibration': revision_root / 'metrics' / 'calibration_ci_oof_final_ema_predictions.json',
    'pooling': revision_root / 'metrics' / 'pooling_sensitivity.csv',
    'baseline': revision_root / 'metrics' / 'baseline_summary.csv',
    'component': revision_root / 'metrics' / 'component_check_summary.json',
    'hrv_domain': revision_root / 'metrics' / 'hrv_domain_summary.csv',
    'robustness': revision_root / 'metrics' / 'robustness_summary.csv',
    'paired_minirocket': revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
    'paired_resnet': revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json',
    'a0_status': revision_root / 'a0_resolution_status.json',
    'claim_map': Path('docs/revision_plan/claim_evidence_map.csv'),
    'task_board': Path('docs/revision_plan/task_board.csv'),
}
optional_final_inputs = {
    'paired_raw_mamba': revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json',
    'paired_transformer': revision_root / 'metrics' / 'paired_full_vs_transformer_comparison.json',
    'paired_hybrid_morphology': revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_comparison.json',
    'hybrid_morphology_summary': revision_root / 'metrics' / 'hybrid_morphology_baseline_summary.json',
    'claim_readiness_gates': revision_root / 'metrics' / 'claim_readiness_gates.json',
    'claim_readiness_table': revision_root / 'tables' / 'table_claim_readiness_gates.csv',
    'external_protocol_gate_summary': revision_root / 'metrics' / 'external_protocol_gate_summary.csv',
    'representation_evidence_status': revision_root / 'metrics' / 'representation_evidence_status.json',
    'representation_probe_summary': revision_root / 'metrics' / 'representation_probe_summary.json',
    'representation_probe_table': revision_root / 'tables' / 'table_representation_probe.csv',
    'representation_cka_table': revision_root / 'tables' / 'table_representation_cka.csv',
    'representation_probe_manifest': revision_root / 'manifests' / 'representation_probe_manifest.json',
    'fewshot_ptbxl_summary': revision_root / 'metrics' / 'fewshot_ptbxl_summary.csv',
    'fewshot_ptbxl_table': revision_root / 'tables' / 'table_fewshot_ptbxl.csv',
    'fewshot_ptbxl_bootstrap': revision_root / 'metrics' / 'fewshot_ptbxl_bootstrap.json',
    'fewshot_ptbxl_manifest': revision_root / 'manifests' / 'fewshot_ptbxl_run_manifest.json',
    'fewshot_cpsc2021_summary': revision_root / 'metrics' / 'fewshot_cpsc2021_summary.csv',
    'fewshot_cpsc2021_table': revision_root / 'tables' / 'table_fewshot_cpsc2021.csv',
    'fewshot_cpsc2021_bootstrap': revision_root / 'metrics' / 'fewshot_cpsc2021_bootstrap.json',
    'fewshot_cpsc2021_manifest': revision_root / 'manifests' / 'fewshot_cpsc2021_run_manifest.json',
    'fewshot_georgia_summary': revision_root / 'metrics' / 'fewshot_georgia_summary.csv',
    'fewshot_georgia_table': revision_root / 'tables' / 'table_fewshot_georgia.csv',
    'fewshot_georgia_bootstrap': revision_root / 'metrics' / 'fewshot_georgia_bootstrap.json',
    'fewshot_georgia_manifest': revision_root / 'manifests' / 'fewshot_georgia_run_manifest.json',
    'robustness_multicomparator_summary': revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
    'robustness_multicomparator_pairwise': revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
    'robustness_multicomparator_table': revision_root / 'tables' / 'table_robustness_multicomparator.csv',
    'robustness_multicomparator_manifest': revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
    'robustness_full_vs_resnet': revision_root / 'metrics' / 'robustness_full_vs_resnet_comparison.json',
    'robustness_full_vs_raw_mamba': revision_root / 'metrics' / 'robustness_full_vs_raw_mamba_comparison.json',
    'robustness_full_vs_transformer': revision_root / 'metrics' / 'robustness_full_vs_transformer_comparison.json',
    'robustness_multicomparator_reviewer_minimal': revision_root / 'metrics' / 'robustness_multicomparator_reviewer_minimal_summary.csv',
    'robustness_multicomparator_reviewer_minimal_pairwise': revision_root / 'metrics' / 'robustness_multicomparator_reviewer_minimal_pairwise.json',
    'robustness_multicomparator_reviewer_minimal_table': revision_root / 'tables' / 'table_robustness_multicomparator_reviewer_minimal.csv',
    'robustness_multicomparator_reviewer_minimal_manifest': revision_root / 'manifests' / 'robustness_multicomparator_reviewer_minimal_manifest.json',
    'robustness_multicomparator_core_final': revision_root / 'metrics' / 'robustness_multicomparator_core_final_summary.csv',
    'robustness_multicomparator_core_final_pairwise': revision_root / 'metrics' / 'robustness_multicomparator_core_final_pairwise.json',
    'robustness_multicomparator_core_final_table': revision_root / 'tables' / 'table_robustness_multicomparator_core_final.csv',
    'robustness_multicomparator_core_final_manifest': revision_root / 'manifests' / 'robustness_multicomparator_core_final_manifest.json',
    'robustness_multicomparator_full_screening': revision_root / 'metrics' / 'robustness_multicomparator_full_screening_summary.csv',
    'robustness_multicomparator_full_screening_pairwise': revision_root / 'metrics' / 'robustness_multicomparator_full_screening_pairwise.json',
    'robustness_multicomparator_full_screening_table': revision_root / 'tables' / 'table_robustness_multicomparator_full_screening.csv',
    'robustness_multicomparator_full_screening_manifest': revision_root / 'manifests' / 'robustness_multicomparator_full_screening_manifest.json',
    'transformer_summary': revision_root / 'metrics' / 'transformer_ecg_baseline_summary.json',
    'transformer_manifest': revision_root / 'manifests' / 'transformer_ecg_baseline_manifest.json',
    'hybrid_morphology_manifest': revision_root / 'manifests' / 'hybrid_morphology_baseline_manifest.json',
    'group_safe_score_calibration_ptbxl': revision_root / 'metrics' / 'group_safe_score_calibration_ptbxl_summary.csv',
    'group_safe_score_calibration_ptbxl_table': revision_root / 'tables' / 'table_group_safe_score_calibration_ptbxl.csv',
    'group_safe_score_calibration_ptbxl_bootstrap': revision_root / 'metrics' / 'group_safe_score_calibration_ptbxl_bootstrap.json',
    'group_safe_score_calibration_ptbxl_splits': revision_root / 'manifests' / 'group_safe_score_calibration_ptbxl_splits.npz',
    'group_safe_score_calibration_ptbxl_coefficients': revision_root / 'tables' / 'table_group_safe_score_calibration_ptbxl_coefficients.csv',
    'group_safe_score_calibration_ptbxl_manifest': revision_root / 'manifests' / 'group_safe_score_calibration_ptbxl_manifest.json',
    'true_fewshot_head_ptbxl': revision_root / 'metrics' / 'true_fewshot_head_ptbxl_summary.csv',
    'true_fewshot_head_ptbxl_table': revision_root / 'tables' / 'table_true_fewshot_head_ptbxl.csv',
    'true_fewshot_head_ptbxl_paired': revision_root / 'tables' / 'table_true_fewshot_head_ptbxl_paired.csv',
    'true_fewshot_head_ptbxl_primary': revision_root / 'tables' / 'table_true_fewshot_head_ptbxl_primary.csv',
    'true_fewshot_head_ptbxl_learning_curve': revision_root / 'tables' / 'table_true_fewshot_head_ptbxl_learning_curve.csv',
    'true_fewshot_head_ptbxl_learning_curve_figure': revision_root / 'figures' / 'figure_true_fewshot_head_ptbxl_learning_curve.png',
    'true_fewshot_head_ptbxl_bootstrap': revision_root / 'metrics' / 'true_fewshot_head_ptbxl_bootstrap.json',
    'true_fewshot_head_ptbxl_coefficients': revision_root / 'tables' / 'table_true_fewshot_head_ptbxl_coefficients.csv',
    'true_fewshot_head_ptbxl_splits': revision_root / 'manifests' / 'true_fewshot_head_ptbxl_splits.npz',
    'true_fewshot_head_manifest': revision_root / 'manifests' / 'true_fewshot_head_ptbxl_manifest.json',
    'external_comparator_paired': revision_root / 'metrics' / 'external_comparator_paired_summary.json',
    'external_comparator_manifest': revision_root / 'manifests' / 'external_comparator_paired_manifest.json',
    'reviewer_completion_contract': revision_root / 'manifests' / 'reviewer_completion_input_contract.json',
    'morphology_learnability_summary': revision_root / 'metrics' / 'morphology_learnability_summary.json',
    'paired_morphology_learnability': revision_root / 'metrics' / 'paired_morphology_learnability_comparison.json',
    'paired_morphology_learnability_table': revision_root / 'tables' / 'table_paired_morphology_learnability.csv',
    'paired_morphology_learnability_manifest': revision_root / 'manifests' / 'paired_morphology_learnability_manifest.json',
    'pooling_sensitivity_external': revision_root / 'metrics' / 'pooling_sensitivity_external.csv',
    'pooling_q3_paired_bootstrap': revision_root / 'metrics' / 'pooling_q3_paired_bootstrap.json',
    'pooling_sensitivity_external_manifest': revision_root / 'manifests' / 'pooling_sensitivity_external_manifest.json',
    'matched_calibration_summary': revision_root / 'metrics' / 'matched_oof_calibration_summary.json',
    'matched_calibration_table': revision_root / 'tables' / 'table_matched_oof_calibration.csv',
    'matched_calibration_paired_table': revision_root / 'tables' / 'table_paired_matched_oof_calibration.csv',
    'matched_calibration_manifest': revision_root / 'manifests' / 'matched_oof_calibration_manifest.json',
    'structured_ablation_summary': revision_root / 'metrics' / 'structured_ablation_5fold_summary.json',
    'structured_ablation_table': revision_root / 'tables' / 'table_structured_ablation_5fold.csv',
    'structured_ablation_paired_table': revision_root / 'tables' / 'table_paired_structured_ablation_5fold.csv',
    'structured_ablation_manifest': revision_root / 'manifests' / 'structured_ablation_5fold_manifest.json',
    'physiological_probe_summary': revision_root / 'metrics' / 'physiological_interval_probe_summary.json',
    'physiological_probe_table': revision_root / 'tables' / 'table_physiological_interval_probe.csv',
    'physiological_probe_manifest': revision_root / 'manifests' / 'physiological_interval_probe_manifest.json',
    'hypothesis_control_ledger': revision_root / 'metrics' / 'hypothesis_control_claim_boundary.json',
    'hypothesis_control_table': revision_root / 'tables' / 'table_hypothesis_control_finding_claim_boundary.csv',
    'marked_manuscript_manifest': revision_root / 'manifests' / 'marked_manuscript_manifest.json',
}
def verify_active_final_inputs_against_mirror(input_map, *, required):
    from scripts.revision.common import sha256_file

    manifest_path = MIRROR_REVISION_ROOT / 'manifests' / 'mirror_manifest.json'
    if not manifest_path.exists() or manifest_path.stat().st_size == 0:
        raise FileNotFoundError(f'Canonical mirror manifest is missing: {manifest_path}')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    rows = {row.get('relative_path'): row for row in manifest.get('artifacts', [])}
    issues = []
    for label, path in input_map.items():
        path = Path(path)
        if not path.exists() or path.stat().st_size == 0:
            if required:
                issues.append(f'{label}:missing={path}')
            continue
        try:
            relative = path.relative_to(revision_root).as_posix()
        except ValueError:
            continue  # Version-controlled planning CSVs are not mirror artifacts.
        row = rows.get(relative)
        if row is None:
            issues.append(f'{label}:unmanifested={relative}')
            continue
        expected_size = int(row.get('size_bytes', -1))
        expected_sha = str(row.get('sha256') or '')
        if expected_size >= 0 and path.stat().st_size != expected_size:
            issues.append(f'{label}:size_mismatch={relative}')
        elif len(expected_sha) != 64 or sha256_file(path) != expected_sha:
            issues.append(f'{label}:sha256_mismatch={relative}')
    if issues:
        raise RuntimeError(
            'Final evidence inputs are not authenticated by the canonical mirror manifest: '
            + '; '.join(issues)
        )
    print(f'Canonical mirror contract verified for {len(input_map)} final-input slots (required={required}).')


def collect_missing_final_inputs(verbose=True):
    missing = []
    for label, path in required_final_inputs.items():
        exists = path.exists()
        if verbose:
            print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')
        if not exists:
            missing.append(f'{label}={path}')
    return missing

missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    print('\nRequired inputs are missing from the active repo. Attempting verified Drive mirror restore before failing.')
    if 'MIRROR_REVISION_ROOT' not in globals():
        raise RuntimeError('MIRROR_REVISION_ROOT is undefined. Run the Setup cell first.')
    if not MIRROR_REVISION_ROOT.exists():
        raise FileNotFoundError(f'Mirror root does not exist: {MIRROR_REVISION_ROOT}')
    run(
        f'python -u scripts/revision/artifact_mirror.py restore '
        f'--mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched '
        f'--exclude-cache-dirs {evidence_restore_args}',
        log_path='reports/revision/logs/final_evidence_required_targeted_restore.log',
    )
    print('\nRechecking required final evidence inputs after mirror restore:')
    missing_inputs = collect_missing_final_inputs(verbose=True)
    if missing_inputs:
        print('\nVerified mirror manifest did not restore every required input; unmanifested Drive files are not accepted.')
if missing_inputs:
    raise FileNotFoundError('Missing required final evidence inputs after checksum-verified mirror restore: ' + '; '.join(missing_inputs))

verify_active_final_inputs_against_mirror(required_final_inputs, required=True)
verify_active_final_inputs_against_mirror(optional_final_inputs, required=False)

# Cross-notebook provenance guard: Notebook 07 must not freeze final tables
# if calibration or paired comparisons still point to an older Full OOF file.
from scripts.revision.common import sha256_file

current_oof_path = revision_root / 'predictions' / 'oof_final_ema_predictions.npz'
current_freeze_path = revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json'
current_oof_sha = sha256_file(current_oof_path)
current_freeze_sha = sha256_file(current_freeze_path)
current_freeze = json.loads(current_freeze_path.read_text(encoding='utf-8'))
print('Current frozen OOF SHA     :', current_oof_sha)
print('Current freeze manifest SHA:', current_freeze_sha)

calibration_payload = json.loads(required_final_inputs['calibration'].read_text(encoding='utf-8'))
calibration_errors = []
if calibration_payload.get('predictions_sha256') != current_oof_sha:
    calibration_errors.append(
        'calibration predictions_sha256 mismatch: '
        f"json={calibration_payload.get('predictions_sha256')} current={current_oof_sha}"
    )
if calibration_payload.get('freeze_manifest_sha256') != current_freeze_sha:
    calibration_errors.append(
        'calibration freeze_manifest_sha256 mismatch: '
        f"json={calibration_payload.get('freeze_manifest_sha256')} current={current_freeze_sha}"
    )
expected_shape = {
    'y_prob': [current_freeze.get('validated_records'), current_freeze.get('n_classes')],
    'y_true': [current_freeze.get('validated_records'), current_freeze.get('n_classes')],
}
if calibration_payload.get('shape') != expected_shape:
    calibration_errors.append(f"calibration shape mismatch: json={calibration_payload.get('shape')} expected={expected_shape}")
if calibration_errors:
    print('\n'.join(' - ' + item for item in calibration_errors))
    raise RuntimeError(
        'Calibration artifact is stale for the current frozen OOF. '
        'Rerun Notebook 03, publish the mirror, then rerun Notebook 04 and Notebook 07.'
    )
print('Calibration provenance guard: OK')

paired_required = {
    'MiniRocket-only': required_final_inputs['paired_minirocket'],
    'ResNet1D/CNN': required_final_inputs['paired_resnet'],
}
paired_optional = {
    'Raw Mamba': optional_final_inputs['paired_raw_mamba'],
    'Transformer ECG': optional_final_inputs['paired_transformer'],
}

def _paired_protocol_compatible(label, payload):
    inputs = payload.get('inputs', {})
    freeze_input = inputs.get('freeze_manifest') or {}
    full_input = inputs.get('full_predictions') or {}
    full_metadata = full_input.get('metadata') or {}
    expected_records = int(current_freeze.get('validated_records', -1))
    expected_classes = int(current_freeze.get('n_classes', -1))
    expected_fingerprint = current_freeze.get('dataset_record_order_fingerprint')
    freeze_contract_ok = (
        freeze_input.get('checkpoint_kind') == 'final_ema'
        and int(freeze_input.get('validated_records', -1)) == expected_records
        and int(freeze_input.get('n_classes', -1)) == expected_classes
        and freeze_input.get('dataset_record_order_fingerprint') == expected_fingerprint
    )
    full_contract_ok = (
        full_metadata.get('checkpoint_kind') == 'final_ema'
        and full_metadata.get('dataset_record_order_fingerprint') == expected_fingerprint
        and full_metadata.get('protocol') == 'fold_final_ema_power_mean_v2_q3_threshold_0.5'
        and float(full_metadata.get('threshold', 0.5)) == 0.5
    )
    current_full_metrics = {}
    current_full_metrics.update(calibration_payload.get('metrics') or {})
    current_full_metrics.update(calibration_payload.get('calibration') or {})
    metric_mismatches = []
    for metric_name, row in (payload.get('metrics') or {}).items():
        if metric_name not in current_full_metrics:
            continue
        observed = row.get('full_value')
        expected = current_full_metrics.get(metric_name)
        if observed is None or expected is None:
            metric_mismatches.append(f'{metric_name}: missing observed={observed} expected={expected}')
            continue
        if abs(float(observed) - float(expected)) > 1e-12:
            metric_mismatches.append(f'{metric_name}: paired={observed} current={expected}')
    return freeze_contract_ok and full_contract_ok and not metric_mismatches, {
        'freeze_contract_ok': freeze_contract_ok,
        'full_contract_ok': full_contract_ok,
        'metric_mismatches': metric_mismatches,
    }


def _check_paired_payload(label, path, required=True):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        if required:
            raise FileNotFoundError(f'Missing required paired comparison for {label}: {path}')
        print(f'Paired {label}: absent/deferred')
        return
    payload = json.loads(path.read_text(encoding='utf-8'))
    inputs = payload.get('inputs', {})
    full_sha = (inputs.get('full_predictions') or {}).get('sha256')
    freeze_sha = (inputs.get('freeze_manifest') or {}).get('sha256')
    if full_sha == current_oof_sha and freeze_sha == current_freeze_sha:
        print(f'Paired {label} provenance guard: OK')
        return
    compatible, details = _paired_protocol_compatible(label, payload)
    if compatible:
        print(
            f'Paired {label} provenance guard: stable-protocol reuse OK. '
            f'Container SHA changed full_sha={full_sha} current={current_oof_sha}; '
            f'freeze_sha={freeze_sha} current={current_freeze_sha}, but final_ema record/class/fingerprint contract '
            'and full metric values match current calibration.'
        )
        return
    raise RuntimeError(
        f'Paired {label} comparison is stale for current OOF. '
        f'full_sha={full_sha} current={current_oof_sha}; '
        f'freeze_sha={freeze_sha} current={current_freeze_sha}; details={details}. '
        'Rerun Notebook 04 paired comparison cells, then rerun Notebook 07.'
    )

for label, path in paired_required.items():
    _check_paired_payload(label, path, required=True)
for label, path in paired_optional.items():
    _check_paired_payload(label, path, required=False)

def restore_optional_final_inputs_from_drive():
    still_missing = [
        f'{label}={path}'
        for label, path in optional_final_inputs.items()
        if not Path(path).exists() or Path(path).stat().st_size == 0
    ]
    print(
        'Optional final-input verified restore status: '
        f'available={len(optional_final_inputs) - len(still_missing)} still_missing={len(still_missing)}'
    )
    if still_missing:
        print('Optional inputs absent from the verified mirror remain deferred:')
        for item in still_missing:
            print(' - missing', item)


restore_optional_final_inputs_from_drive()

print('Optional final evidence inputs:')
for label, path in optional_final_inputs.items():
    exists = path.exists()
    print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')

# Guard against stale Colab code through the generator's stable capability contract.
# Do not inspect internal helper names: those may change during valid refactors.
robustness_profile_audit_path = Path('scripts/revision/robustness_profile_audit.py')
if not robustness_profile_audit_path.exists() or 'def select_best_profile' not in robustness_profile_audit_path.read_text(encoding='utf-8'):
    raise RuntimeError('Missing/stale robustness profile validator. Pull latest main before running Notebook 07.')
generator_path = Path('scripts/revision/13_final_evidence_matrix.py')
generator_source = generator_path.read_text(encoding='utf-8')
import ast
generator_tree = ast.parse(generator_source, filename=str(generator_path))
generator_contract = {}
for node in generator_tree.body:
    if not isinstance(node, (ast.Assign, ast.AnnAssign)):
        continue
    targets = node.targets if isinstance(node, ast.Assign) else [node.target]
    value = node.value
    for target in targets:
        if isinstance(target, ast.Name) and target.id in {'FINAL_EVIDENCE_SCHEMA_VERSION', 'FINAL_EVIDENCE_CAPABILITIES'}:
            generator_contract[target.id] = ast.literal_eval(value)
required_generator_schema = 12
required_generator_capabilities = {
    'adaptation_learning_curve',
    'claim_readiness_gates',
    'external_learned_comparator_audit',
    'group_safe_score_calibration_v2',
    'hybrid_morphology_paired',
    'learned_comparator_robustness_audit',
    'matched_cross_fitted_calibration',
    'authenticated_matched_calibration_v5',
    'matched_structured_ablation_5fold',
    'matched_structured_ablation_fresh_full',
    'post_initial_review_adaptation_analysis_lock',
    'physiological_interval_probe_gate',
    'physiological_interval_probe_v3',
    'representation_probe_v3',
    'reviewer_presentation_assets',
    'reviewer_gap_closure_v1',
    'morphology_kernel_learnability_control',
    'external_zero_target_group_paired_ci',
    'pooling_q3_cross_dataset_sensitivity',
    'transformer_paired',
    'true_fewshot_frozen_encoder_head_v2',
}
generator_schema = int(generator_contract.get('FINAL_EVIDENCE_SCHEMA_VERSION', 0))
generator_capabilities = set(generator_contract.get('FINAL_EVIDENCE_CAPABILITIES', ()))
missing_generator_capabilities = sorted(required_generator_capabilities - generator_capabilities)
if generator_schema < required_generator_schema or missing_generator_capabilities:
    raise RuntimeError(
        'Final evidence generator is stale and cannot ingest the current optional evidence artifacts. '
        f'required_schema={required_generator_schema} observed_schema={generator_schema}; '
        f'missing_capabilities={missing_generator_capabilities}. '
        'Pull latest main or rerun this notebook direct setup guard, then rerun Notebook 07.'
    )
print(
    'Final evidence generator capability contract: OK | '
    f'schema={generator_schema} capabilities={sorted(generator_capabilities)}'
)

fewshot_datasets_for_final = ['ptbxl', 'cpsc2021', 'georgia']
fewshot_complete_datasets = []
fewshot_artifacts_present = False
for dataset in fewshot_datasets_for_final:
    dataset_paths = [
        optional_final_inputs[f'fewshot_{dataset}_summary'],
        optional_final_inputs[f'fewshot_{dataset}_table'],
        optional_final_inputs[f'fewshot_{dataset}_bootstrap'],
        optional_final_inputs[f'fewshot_{dataset}_manifest'],
    ]
    dataset_artifacts_present = all(path.exists() and path.stat().st_size > 0 for path in dataset_paths)
    print(f'Few-shot artifact set complete for {dataset}:', dataset_artifacts_present)
    if dataset_artifacts_present:
        fewshot_manifest = json.loads(optional_final_inputs[f'fewshot_{dataset}_manifest'].read_text(encoding='utf-8'))
        print(f'Few-shot manifest status for {dataset}:', fewshot_manifest.get('status'))
        if fewshot_manifest.get('status') != 'complete':
            raise RuntimeError(f'Few-shot manifest exists for {dataset} but is not complete; rerun Notebook 02 few-shot cell or remove stale artifact.')
        fewshot_complete_datasets.append(dataset)
fewshot_artifacts_present = bool(fewshot_complete_datasets)
print('Few-shot complete datasets:', fewshot_complete_datasets)

paired_resnet = json.loads(required_final_inputs['paired_resnet'].read_text(encoding='utf-8'))
resnet_metrics = paired_resnet.get('metrics', {})
print('Paired ResNet interpretations:')
for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
    row = resnet_metrics.get(metric, {})
    print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")

paired_raw_path = optional_final_inputs['paired_raw_mamba']
if paired_raw_path.exists():
    paired_raw = json.loads(paired_raw_path.read_text(encoding='utf-8'))
    raw_metrics = paired_raw.get('metrics', {})
    print('Paired Raw Mamba interpretations:')
    for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
        row = raw_metrics.get(metric, {})
        print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")
else:
    print('Paired Raw Mamba optional input is absent; final matrix will keep Raw Mamba incomplete until Notebook 04 produces it.')

expected_external_datasets = ['ptbxl', 'georgia', 'cpsc2021']
external_gate_path = optional_final_inputs['external_protocol_gate_summary']
if external_gate_path.exists():
    external_gate = pd.read_csv(external_gate_path)
    external_gate_columns = [
        'dataset',
        'status',
        'protocol_gate_passed',
        'manuscript_ready',
        'issue_count',
        'reused_existing',
        'gate_cache_key',
        'prediction_sha256',
        'slice_prediction_sha256',
        'gate_json',
        'gate_manifest',
    ]
    display_columns = [col for col in external_gate_columns if col in external_gate.columns]
    print('External protocol gate summary:')
    display(external_gate[display_columns])
    gate_pass_mask = external_gate.get('protocol_gate_passed', pd.Series(False, index=external_gate.index)).astype(str).str.lower().isin({'true', '1', 'yes'})
    present_datasets = sorted(external_gate['dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    passed_datasets = sorted(external_gate.loc[gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    blocked_datasets = sorted(external_gate.loc[~gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    deferred_datasets = [dataset for dataset in expected_external_datasets if dataset not in present_datasets]
    print('External gate passed datasets:', ', '.join(passed_datasets) if passed_datasets else 'none')
    print('External gate blocked datasets in summary:', ', '.join(blocked_datasets) if blocked_datasets else 'none')
    print('External deferred datasets absent from gate summary:', ', '.join(deferred_datasets) if deferred_datasets else 'none')
    print('External gate supporting artifacts:')
    for dataset in sorted(external_gate['dataset'].astype(str).tolist()) if 'dataset' in external_gate.columns else []:
        supporting = {
            'gate_json': revision_root / 'metrics' / f'external_{dataset}_protocol_gate.json',
            'label_mapping': revision_root / 'tables' / f'table_external_{dataset}_label_mapping.csv',
            'metrics_table': revision_root / 'tables' / f'table_external_{dataset}_metrics.csv',
            'gate_manifest': revision_root / 'manifests' / f'external_{dataset}_protocol_gate_manifest.json',
        }
        for label, path in supporting.items():
            print(f'  {dataset:8} {label:14}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
else:
    print('External protocol gate summary absent; external datasets remain experimental/deferred in final evidence.')

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## Claim Readiness Gates

Generate a lightweight blocker ledger for optional or unsupported claims before building the final matrix. This keeps full-HRV, clinical-readiness, broad-superiority, transformer, hybrid, and learned-comparator robustness wording auditable.


In [ ]:
# Refresh legacy calibration provenance, build reviewer-facing assets, and then run claim gates.
# This cell is CPU-only and can be rerun safely; current artifacts are reused when their contracts match.
calibration_path = revision_root / 'metrics' / 'calibration_ci_oof_final_ema_predictions.json'
calibration_prediction_path = revision_root / 'predictions' / 'oof_final_ema_predictions.npz'
calibration_freeze_path = revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json'
calibration_payload = json.loads(calibration_path.read_text(encoding='utf-8'))
calibration_bootstrap = calibration_payload.get('bootstrap') or {}
FORENSIC_BOOTSTRAP_BINDING_CHECK = 'freeze_group_sidecar_v1'
EXPECTED_GROUP_REFERENCE = 'https://physionet.org/content/ecg-arrhythmia/1.0.0/'
calibration_refresh_reasons = []
if calibration_payload.get('n_boot', 0) < 1000:
    calibration_refresh_reasons.append(f"n_boot={calibration_payload.get('n_boot')}<1000")
if calibration_bootstrap.get('unit') != 'authenticated_source_patient_record':
    calibration_refresh_reasons.append(
        f"bootstrap.unit={calibration_bootstrap.get('unit')!r}"
    )
if calibration_bootstrap.get('independence_contract') != 'physionet_ecg_arrhythmia_one_patient_per_record_v1':
    calibration_refresh_reasons.append(
        'bootstrap.independence_contract='
        f"{calibration_bootstrap.get('independence_contract')!r}"
    )
if calibration_bootstrap.get('group_semantics_reference') != EXPECTED_GROUP_REFERENCE:
    calibration_refresh_reasons.append('bootstrap.group_semantics_reference_missing_or_mismatched')
if not calibration_bootstrap.get('group_sidecar_sha256'):
    calibration_refresh_reasons.append('bootstrap.group_sidecar_sha256_missing')
if calibration_refresh_reasons:
    print('Refreshing calibration CI provenance before reviewer-asset generation:')
    for reason in calibration_refresh_reasons:
        print(' -', reason)
    run(
        'python -u scripts/revision/04_calibration_ci.py '
        f'--predictions "{calibration_prediction_path}" --out "{calibration_path}" '
        f'--freeze-manifest "{calibration_freeze_path}" --require-manuscript-ready '
        '--threshold 0.5 --n-bins 15 --n-boot 1000 --seed 42',
        log_path='reports/revision/logs/final_evidence_calibration_contract_refresh.log',
    )
    calibration_payload = json.loads(calibration_path.read_text(encoding='utf-8'))
    calibration_bootstrap = calibration_payload.get('bootstrap') or {}
    if (
        calibration_bootstrap.get('unit') != 'authenticated_source_patient_record'
        or calibration_bootstrap.get('independence_contract') != 'physionet_ecg_arrhythmia_one_patient_per_record_v1'
        or calibration_bootstrap.get('group_semantics_reference') != EXPECTED_GROUP_REFERENCE
        or not calibration_bootstrap.get('group_sidecar_sha256')
    ):
        raise RuntimeError('Calibration CI refresh completed without the required subject-level provenance contract.')
    run(
        'python -u scripts/revision/05_artifact_inventory.py',
        log_path='reports/revision/logs/final_evidence_calibration_refresh_inventory.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
        f'--source-conflict-policy newer --refresh-existing-cache-dirs --mirror-root "{MIRROR_REVISION_ROOT}"',
        log_path='reports/revision/logs/final_evidence_calibration_refresh_mirror_publish.log',
    )
else:
    print('Calibration CI subject-level bootstrap provenance: reusable')

presentation_script = Path('scripts/revision/29_reviewer_presentation_assets.py')
presentation_required_tokens = [
    'authenticated_source_patient_record',
    'PRESENTATION_CONTRACT_SCHEMA_VERSION = 2',
    'AUTHENTICATED_RECORD_BOOTSTRAP_UNIT',
    'CHAPMAN_GROUP_SEMANTICS',
    'table_morphology_transform_contract.csv',
    'table_calibration_ci_compact.csv',
    'table_paired_baseline_ci_compact.csv',
    'table_pooling_sensitivity_compact.csv',
    'table_fold_pca_provenance.csv',
    'table_training_configuration.csv',
]
presentation_source = presentation_script.read_text(encoding='utf-8') if presentation_script.exists() else ''
missing_presentation_tokens = [
    token for token in presentation_required_tokens if token not in presentation_source
]
if missing_presentation_tokens:
    raise RuntimeError(
        'Reviewer presentation asset generator is stale. Pull latest main. Missing tokens: '
        f'{missing_presentation_tokens}'
    )
run(
    'python -u scripts/revision/29_reviewer_presentation_assets.py --strict',
    log_path='reports/revision/logs/reviewer_presentation_assets.log',
)

# R1-C6 self-heal: use the strict closure validator itself as the preflight, then
# regenerate the CPU-only external pooling package only when its current contract fails.
# This never reruns model inference; it aggregates frozen slice probabilities.
import importlib.util
from types import SimpleNamespace

pooling_gap_datasets = ('ptbxl', 'georgia', 'cpsc2021')
pooling_gap_args = SimpleNamespace(
    pooling_table=revision_root / 'tables' / 'table_pooling_sensitivity_across_datasets.csv',
    pooling_bootstrap=revision_root / 'metrics' / 'pooling_q3_paired_bootstrap.json',
    pooling_manifest=revision_root / 'manifests' / 'pooling_sensitivity_external_manifest.json',
)

def _reviewer_gap_pooling_status():
    module_path = Path('scripts/revision/41_reviewer_gap_closure.py')
    spec = importlib.util.spec_from_file_location('_reviewer_gap_pooling_preflight', module_path)
    if spec is None or spec.loader is None:
        return {'reviewer_item': 'R1-C6', 'status': 'incomplete', 'issues': ['validator_import_failed']}
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    try:
        status, _compact = module.validate_pooling(pooling_gap_args)
        return status
    except Exception as exc:
        return {
            'reviewer_item': 'R1-C6',
            'status': 'incomplete',
            'issues': [f'pooling_preflight_exception={type(exc).__name__}: {exc}'],
        }

pooling_gap_status = _reviewer_gap_pooling_status()
print('R1-C6 pooling preflight:', json.dumps(pooling_gap_status, indent=2))
if pooling_gap_status.get('status') != 'complete':
    pooling_source_paths = []
    for dataset in pooling_gap_datasets:
        external_root = revision_root / 'experimental' / 'external' / dataset
        pooling_source_paths.extend([
            revision_root / 'metrics' / f'external_{dataset}_protocol_gate.json',
            external_root / f'{dataset}_full_predictions.npz',
            external_root / f'{dataset}_full_slice_predictions.npz',
        ])
    missing_pooling_sources = [
        str(path) for path in pooling_source_paths
        if not path.exists() or path.stat().st_size == 0
    ]
    if missing_pooling_sources:
        raise FileNotFoundError(
            'Cannot refresh R1-C6 pooling evidence because authenticated external inputs are missing: '
            + '; '.join(missing_pooling_sources)
        )
    pooling_cache_dir = MIRROR_REVISION_ROOT / 'metrics' / 'pooling_external_metric_cache'
    run(
        'python -u scripts/revision/30_pooling_sensitivity_external.py '
        '--dataset ptbxl --dataset georgia --dataset cpsc2021 '
        '--q-values 1,2,3,4,8,max --threshold 0.5 --n-bins 15 '
        '--n-boot 1000 --seed 42 --strict-group-bootstrap --reuse-existing '
        f'--metric-cache-dir "{pooling_cache_dir}"',
        log_path='reports/revision/logs/reviewer_gap_pooling_refresh.log',
    )
    pooling_gap_status = _reviewer_gap_pooling_status()
    print('R1-C6 pooling post-refresh:', json.dumps(pooling_gap_status, indent=2))
    if pooling_gap_status.get('status') != 'complete':
        raise RuntimeError(
            'R1-C6 pooling regeneration finished but the strict contract is still incomplete: '
            + '; '.join(pooling_gap_status.get('issues') or ['unknown issue'])
        )
    run(
        'python -u scripts/revision/05_artifact_inventory.py',
        log_path='reports/revision/logs/reviewer_gap_pooling_refresh_inventory.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
        f'--source-conflict-policy source --refresh-existing-cache-dirs '
        f'--refresh-existing-prefix metrics/pooling_external_metric_cache '
        f'--mirror-root "{MIRROR_REVISION_ROOT}"',
        log_path='reports/revision/logs/reviewer_gap_pooling_refresh_mirror_publish.log',
    )
else:
    print('R1-C6 external pooling package is current; reusing the authenticated 18-row grid.')

# Authenticate and present the four reviewer-gap closure packages before claim readiness.
reviewer_gap_script = Path('scripts/revision/41_reviewer_gap_closure.py')
reviewer_gap_required_tokens = [
    'R1-C2', 'R1-C5', 'R1-C6', 'R2-C3',
    'table_external_zero_target_ci_compact.csv',
    'table_pooling_cross_dataset_compact.csv',
    'table_morphology_learnability_compact.csv',
    'table_robustness_six_stress_compact.csv',
]
reviewer_gap_source = reviewer_gap_script.read_text(encoding='utf-8') if reviewer_gap_script.exists() else ''
missing_reviewer_gap_tokens = [token for token in reviewer_gap_required_tokens if token not in reviewer_gap_source]
if missing_reviewer_gap_tokens:
    raise RuntimeError(
        'Reviewer gap-closure generator is stale. Pull latest main. Missing tokens: '
        f'{missing_reviewer_gap_tokens}'
    )
run(
    'python -u scripts/revision/41_reviewer_gap_closure.py --strict',
    log_path='reports/revision/logs/reviewer_gap_closure.log',
)
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/reviewer_gap_closure_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
    f'--source-conflict-policy source --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/reviewer_gap_closure_mirror_publish.log',
)


# No embedded stale fallback: the repository script itself is the auditable source.
claim_readiness_script = Path('scripts/revision/28_claim_readiness_gates.py')
required_tokens = [
    'group_safe_score_calibration_v2',
    'true_fewshot_frozen_encoder_head_adaptation',
    'external_learned_comparator_audit',
    'representation_probe_fold_safe_v3_projection_and_fold_audit',
    'marked_highlighted_manuscript',
    'robustness_contract_issues',
    'ROBUSTNESS_N_BOOT',
    'adaptation_grid_issues',
    'ADAPTATION_PRIMARY_FRACTION_POLICY',
    'manifest_output_issues',
    'primary_endpoint_issues',
    'morphology_kernel_learnability_control',
    'external_zero_target_label_paired_ci',
    'pooling_q3_cross_dataset_sensitivity',
    'pooling_sensitivity_external.csv',
    'reviewer_gap_closure_status.json',
]
source_text = claim_readiness_script.read_text(encoding='utf-8') if claim_readiness_script.exists() else ''
missing_tokens = [token for token in required_tokens if token not in source_text]
if missing_tokens:
    raise RuntimeError(
        'Claim-readiness gate is stale. Pull/push the current main branch or copy the updated script into the active repo. '
        f'Missing tokens: {missing_tokens}'
    )
run('python -u scripts/revision/28_claim_readiness_gates.py', log_path='reports/revision/logs/claim_readiness_gates.log')
for item in [
    revision_root / 'metrics' / 'claim_readiness_gates.json',
    revision_root / 'tables' / 'table_claim_readiness_gates.csv',
    revision_root / 'manifests' / 'claim_readiness_gates_manifest.json',
]:
    print(f'{item}: exists={item.exists()} size={item.stat().st_size if item.exists() else None}')


## Final Evidence Matrix

Build claim-level evidence, blocker status, robustness claim rows, and reviewer-safe wording from frozen artifacts.

In [ ]:
import pandas as pd

run('python -u scripts/revision/13_final_evidence_matrix.py --strict', log_path='reports/revision/logs/final_evidence_matrix.log')
matrix = pd.read_csv('reports/revision/tables/table_final_evidence_matrix.csv')
safe = pd.read_csv('reports/revision/tables/table_final_safe_wording.csv')
robust = pd.read_csv('reports/revision/tables/table_final_robustness_claims.csv')
blockers = pd.read_csv('reports/revision/tables/table_final_blocker_status.csv')
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
guidance = payload.get('claim_guidance', {})
for token in ['claim_readiness_gates', 'hybrid_morphology', 'fewshot', 'external_comparators', 'marked_manuscript', 'robustness', 'reviewer_gap_closure', 'morphology_learnability', 'external_zero_target_ci', 'pooling_cross_dataset']:
    if token not in guidance:
        raise RuntimeError(f'Final generator is stale; missing claim guidance token: {token}')
legacy = payload.get('legacy_row_split_score_calibration', {})
if any(item.get('claim_ready') is True for item in legacy.values() if isinstance(item, dict)):
    raise RuntimeError('Legacy row-split score-calibration artifact was incorrectly marked claim-ready.')
print('Final evidence matrix validated. final_ready_for_rebuttal=', payload.get('final_ready_for_rebuttal'))
print('Group-safe score calibration:', payload.get('group_safe_score_calibration_summary', {}))
print('True frozen-encoder head adaptation:', payload.get('true_fewshot_head_adaptation_summary', {}))
print('External learned-comparator audit:', payload.get('external_learned_comparator_audit', {}))
learned_robustness_audit = payload.get('learned_comparator_robustness_audit', {})
selected_robustness_profile = learned_robustness_audit.get('selected') or {}
print('Selected learned-comparator robustness audit:', learned_robustness_audit)
if learned_robustness_audit.get('complete') and not selected_robustness_profile.get('valid'):
    raise RuntimeError('Final generator selected an invalid learned-comparator robustness profile.')
if selected_robustness_profile.get('evidence_tier') == 'screening_subset' and selected_robustness_profile.get('metric_specific_ci_ready'):
    raise RuntimeError('A screening robustness profile was incorrectly marked final-CI ready.')
readiness_rows = {
    row.get('claim_id'): row
    for row in (payload.get('claim_readiness_gates', {}).get('rows') or [])
    if isinstance(row, dict)
}
learned_robustness_readiness = readiness_rows.get('robustness_learned_comparators', {})
print('Learned-comparator robustness readiness:', learned_robustness_readiness)
if learned_robustness_readiness.get('manuscript_ready') is True:
    if learned_robustness_readiness.get('status') != 'complete_multicomparator_robustness_available' or not selected_robustness_profile.get('canonical_gate_ready'):
        raise RuntimeError('Learned-comparator robustness was marked manuscript-ready without the canonical complete gate.')
if selected_robustness_profile.get('canonical_gate_ready') and learned_robustness_readiness.get('manuscript_ready') is not True:
    raise RuntimeError('Canonical learned-comparator robustness is complete but the claim-readiness gate did not recognize it.')
reviewer_gap_closure = payload.get('reviewer_gap_closure') or {}
reviewer_gap_rows = {row.get('reviewer_item'): row for row in reviewer_gap_closure.get('rows', []) if isinstance(row, dict)}
expected_reviewer_gap_items = {'R1-C2', 'R1-C5', 'R1-C6', 'R2-C3'}
if reviewer_gap_closure.get('status') is not True or set(reviewer_gap_rows) != expected_reviewer_gap_items:
    raise RuntimeError('Final evidence does not contain the complete four-item reviewer gap-closure package.')
if any(row.get('manuscript_ready') is not True or row.get('status') != 'complete' for row in reviewer_gap_rows.values()):
    raise RuntimeError('At least one reviewer gap-closure item is not manuscript-ready.')
print('Reviewer gap closure:', reviewer_gap_rows)
display(matrix[['claim_id', 'claim_topic', 'evidence_status', 'key_numbers', 'blocker']])
display(safe[['claim_id', 'evidence_status', 'safe_wording']])
display(robust[['stress_test', 'metric', 'degradation_interpretation', 'stressed_performance_interpretation']])
display(blockers[['blocker_id', 'resolution_status', 'restriction']])


## Hypothesis-Control-Finding-Claim Boundary Ledger


In [ ]:
# CPU-only synthesis. Strict mode becomes the final gate after Notebook 03/04 and PTB-XL learning-curve rerun.
HYPOTHESIS_LEDGER_STRICT = os.environ.get('ECG_RAMBA_HYPOTHESIS_LEDGER_STRICT', '1') == '1'

hypothesis_command = 'python -u scripts/revision/45_hypothesis_control_claim_boundary.py'
if HYPOTHESIS_LEDGER_STRICT:
    hypothesis_command += ' --strict'
run(
    hypothesis_command,
    log_path='reports/revision/logs/hypothesis_control_claim_boundary.log',
)
hypothesis_table = Path('reports/revision/tables/table_hypothesis_control_finding_claim_boundary.csv')
hypothesis_tex = Path('reports/revision/tables/table_hypothesis_control_finding_claim_boundary.tex')
hypothesis_json = Path('reports/revision/metrics/hypothesis_control_claim_boundary.json')
if not hypothesis_table.is_file() or not hypothesis_tex.is_file() or not hypothesis_json.is_file():
    raise RuntimeError('Hypothesis-control-finding-claim-boundary outputs are missing.')
display(pd.read_csv(hypothesis_table))
print(json.dumps(json.loads(hypothesis_json.read_text(encoding='utf-8')), indent=2)[:8000])

# Refresh the final matrix once so its top-level payload authenticates the ledger
# produced immediately above. The generator keeps incomplete optional experiments
# as claim-specific blockers rather than failing the base rebuttal package.
run(
    'python -u scripts/revision/13_final_evidence_matrix.py --strict',
    log_path='reports/revision/logs/final_evidence_matrix_after_hypothesis_ledger.log',
)


run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
    f'--refresh-existing-prefix metrics/hypothesis_control_claim_boundary.json '
    f'--refresh-existing-prefix tables/table_hypothesis_control_finding_claim_boundary. '
    f'--refresh-existing-prefix manifests/hypothesis_control_claim_boundary_manifest.json '
    f'--refresh-existing-prefix metrics/final_evidence_matrix.json '
    f'--refresh-existing-prefix tables/table_final_ '
    f'--refresh-existing-prefix manifests/final_evidence_matrix_manifest.json '
    f'--mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/hypothesis_control_claim_boundary_mirror_publish.log',
)


## Validation Gate

Fail if the final evidence matrix is incomplete or if blocked/limited claims are accidentally marked as fully supported.

In [ ]:
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('reports/revision/manifests/final_evidence_matrix_manifest.json').read_text(encoding='utf-8'))
required = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
    Path('reports/revision/metrics/reviewer_gap_closure_status.json'),
    Path('reports/revision/tables/table_reviewer_gap_closure_status.csv'),
    Path('reports/revision/manifests/reviewer_gap_closure_manifest.json'),
    Path('reports/revision/tables/table_external_zero_target_ci_compact.csv'),
    Path('reports/revision/tables/table_pooling_cross_dataset_compact.csv'),
    Path('reports/revision/tables/table_morphology_learnability_compact.csv'),
    Path('reports/revision/tables/table_robustness_six_stress_compact.csv'),
]
missing = [str(path) for path in required if not path.exists() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError('Missing final evidence outputs: ' + '; '.join(missing))
if payload.get('contract_issues'):
    raise RuntimeError('Final evidence has stale contracts: ' + '; '.join(payload['contract_issues']))
if payload.get('missing_inputs'):
    raise RuntimeError('Final evidence has missing inputs: ' + '; '.join(payload['missing_inputs']))
if manifest.get('final_ready_for_rebuttal') is not True or payload.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence is not ready for rebuttal use; inspect final evidence blockers.')
required_claim_ids = {'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07'}
observed_claim_ids = {row.get('claim_id') for row in payload.get('evidence_matrix', [])}
if observed_claim_ids != required_claim_ids:
    raise RuntimeError(f'Final evidence claim IDs are unexpected: {sorted(observed_claim_ids)}')
if len(payload.get('robustness_claims', [])) != 30:
    raise RuntimeError('Final robustness matrix must contain 30 stress/metric rows.')
c04 = next((row for row in payload['evidence_matrix'] if row.get('claim_id') == 'C04'), {})
if c04.get('evidence_status') == 'complete_probe_available_with_disentanglement_limitation':
    if 'do not support label-aligned morphology-rhythm disentanglement' not in c04.get('safe_wording', ''):
        raise RuntimeError('Representation safe wording is too strong.')
elif 'representation separation remains unproven' not in c04.get('key_numbers', ''):
    raise RuntimeError('Representation state is ambiguous.')
print('Final evidence validation gate: PASS')


## Inventory And Stable Mirror

Refresh the artifact manifest and publish final evidence outputs back to Drive.

In [ ]:
FORENSIC_SUBMISSION_DOCUMENT_GATE = 'current_document_sha_bound_v1'
import shlex
import shutil
from datetime import datetime, timezone
from scripts.revision.common import save_json, save_csv, sha256_file

SUBMISSION_MODE = os.environ.get('ECG_RAMBA_SUBMISSION_MODE', 'snapshot').strip().lower()
if SUBMISSION_MODE not in {'snapshot', 'final'}:
    raise ValueError('ECG_RAMBA_SUBMISSION_MODE must be snapshot or final.')
REQUIRE_SUBMISSION_PACKAGE = SUBMISSION_MODE == 'final'
submission_inputs = {
    'original_tex': Path(os.environ.get('ECG_RAMBA_ORIGINAL_TEX', '')).expanduser(),
    'revised_tex': Path(os.environ.get('ECG_RAMBA_REVISED_TEX', '')).expanduser(),
    'response': Path(os.environ.get('ECG_RAMBA_RESPONSE_PATH', '')).expanduser(),
}
missing_submission_inputs = [
    name for name, path in submission_inputs.items()
    if not str(path) or not path.is_file() or path.stat().st_size == 0
]
submission_status_path = Path('reports/revision/metrics/submission_package_readiness.json')
submission_status_table = Path('reports/revision/tables/table_submission_package_readiness.csv')
if missing_submission_inputs:
    submission_status = {
        'status': 'blocked_missing_current_document_inputs',
        'submission_package_ready': False,
        'evidence_snapshot_may_continue': True,
        'submission_mode': SUBMISSION_MODE,
        'missing_inputs': missing_submission_inputs,
        'required_environment': {
            'ECG_RAMBA_ORIGINAL_TEX': 'previous submitted manuscript TeX',
            'ECG_RAMBA_REVISED_TEX': 'current revised main.tex',
            'ECG_RAMBA_RESPONSE_PATH': 'current response letter source/text',
        },
        'created_utc': datetime.now(timezone.utc).isoformat(),
    }
    save_json(submission_status_path, submission_status)
    save_csv(submission_status_table, [{
        'status': submission_status['status'],
        'submission_package_ready': False,
        'missing_inputs': ','.join(missing_submission_inputs),
    }])
    print('Submission document gate:', submission_status['status'])
    if REQUIRE_SUBMISSION_PACKAGE:
        raise RuntimeError(
            'Submission package was required but current document inputs are missing: '
            + ', '.join(missing_submission_inputs)
        )
else:
    marked_dir = Path('reports/revision/manuscript')
    marked_dir.mkdir(parents=True, exist_ok=True)
    if shutil.which('latexmk') is None or shutil.which('pdftotext') is None:
        raise RuntimeError('Final document gate requires latexmk and pdftotext in PATH.')
    revised_pdf = marked_dir / 'main_revised.pdf'
    revised_pdf_text = marked_dir / 'main_revised.pdf.txt'
    revised_compile_dir = marked_dir.resolve()
    run(
        'latexmk -pdf -interaction=nonstopmode -halt-on-error '
        f'-outdir={shlex.quote(str(revised_compile_dir))} '
        f'{shlex.quote(str(submission_inputs["revised_tex"].resolve()))}',
        cwd=submission_inputs['revised_tex'].resolve().parent,
        log_path='reports/revision/logs/final_revised_manuscript_compile.log',
    )
    produced_pdf = revised_compile_dir / submission_inputs['revised_tex'].with_suffix('.pdf').name
    if not produced_pdf.is_file() or produced_pdf.stat().st_size == 0:
        raise FileNotFoundError(f'latexmk did not produce the revised PDF: {produced_pdf}')
    if produced_pdf.resolve() != revised_pdf.resolve():
        shutil.copy2(produced_pdf, revised_pdf)
    run(
        f'pdftotext {shlex.quote(str(revised_pdf.resolve()))} {shlex.quote(str(revised_pdf_text.resolve()))}',
        log_path='reports/revision/logs/final_revised_manuscript_pdftotext.log',
    )
    if not revised_pdf_text.is_file() or revised_pdf_text.stat().st_size == 0:
        raise RuntimeError('pdftotext produced no text for the current revised PDF.')
    marked_tex = marked_dir / 'main_marked.tex'
    marked_pdf = marked_dir / 'main_marked.pdf'
    marked_manifest = Path('reports/revision/manifests/marked_manuscript_manifest.json')
    run(
        'python -u scripts/revision/36_build_marked_manuscript.py '
        f"--original {shlex.quote(str(submission_inputs['original_tex']))} "
        f"--revised {shlex.quote(str(submission_inputs['revised_tex']))} "
        f"--out-tex {shlex.quote(str(marked_tex))} "
        f"--out-pdf {shlex.quote(str(marked_pdf))} "
        f"--out-manifest {shlex.quote(str(marked_manifest))} --compile --strict",
        log_path='reports/revision/logs/final_marked_manuscript_build.log',
    )
    scan_paths = [
        submission_inputs['revised_tex'],
        submission_inputs['response'],
        revised_pdf_text,
        Path('reports/revision/tables'),
    ]
    scan_command = 'python -u scripts/revision/46_submission_claim_scan.py --strict '
    scan_command += ' '.join(f"--path {shlex.quote(str(path))}" for path in scan_paths)
    scan_command += (
        ' --out-json reports/revision/metrics/submission_forbidden_claim_scan.json'
        ' --out-table reports/revision/tables/table_submission_forbidden_claim_scan.csv'
    )
    run(scan_command, log_path='reports/revision/logs/final_submission_forbidden_claim_scan.log')
    scan_payload = json.loads(
        Path('reports/revision/metrics/submission_forbidden_claim_scan.json').read_text(encoding='utf-8')
    )
    marked_payload = json.loads(marked_manifest.read_text(encoding='utf-8'))
    submission_status = {
        'status': 'complete' if scan_payload.get('status') is True and marked_payload.get('editorial_ready') is True else 'blocked',
        'submission_package_ready': bool(
            scan_payload.get('status') is True and marked_payload.get('editorial_ready') is True
        ),
        'submission_mode': SUBMISSION_MODE,
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'input_sha256': {
            name: sha256_file(path) for name, path in submission_inputs.items()
        },
        'compiled_revised_pdf': {
            'path': str(revised_pdf),
            'sha256': sha256_file(revised_pdf),
            'text_path': str(revised_pdf_text),
            'text_sha256': sha256_file(revised_pdf_text),
        },
        'forbidden_claim_scan': {
            'path': 'reports/revision/metrics/submission_forbidden_claim_scan.json',
            'sha256': sha256_file('reports/revision/metrics/submission_forbidden_claim_scan.json'),
        },
        'marked_manuscript_manifest': {
            'path': str(marked_manifest),
            'sha256': sha256_file(marked_manifest),
        },
    }
    save_json(submission_status_path, submission_status)
    save_csv(submission_status_table, [{
        'status': submission_status['status'],
        'submission_package_ready': submission_status['submission_package_ready'],
        'missing_inputs': '',
    }])
    if not submission_status['submission_package_ready']:
        raise RuntimeError('Current manuscript/response/PDF submission gate did not pass.')
    print('Submission document gate: PASS')


In [ ]:
FORENSIC_NOTEBOOK07_FINAL_GATE = 'strict_full_sha_authority_update_v3'
if CODE_AUTHORITY.get('git_commit') != os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT'):
    raise RuntimeError('Notebook 07 code-authority session contract is missing or inconsistent.')
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/final_artifact_inventory.log',
)
# Current-authority outputs may legitimately replace an older canonical audit.
# The detached commit pin above prevents a stale branch/runtime from publishing.
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing full --source-conflict-policy source --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/final_evidence_authority_publish.log',
)
run(
    f'python -u scripts/revision/38_pipeline_storage_audit.py --canonical-root "{MIRROR_REVISION_ROOT}" --strict --full-sha '
    '--out-json reports/revision/metrics/pipeline_storage_audit.json '
    '--out-csv reports/revision/tables/table_pipeline_storage_audit.csv',
    log_path='reports/revision/logs/final_pipeline_storage_audit_strict_full_sha.log',
)
run(
    f'python -u scripts/revision/47_forensic_notebook_audit.py --canonical-root "{MIRROR_REVISION_ROOT}" --strict',
    log_path='reports/revision/logs/final_notebook_forensic_audit.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing full --source-conflict-policy source --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/final_forensic_audit_authority_publish.log',
)
run(
    f'python -u scripts/revision/38_pipeline_storage_audit.py --canonical-root "{MIRROR_REVISION_ROOT}" --strict --full-sha '
    '--out-json reports/revision/metrics/pipeline_storage_audit.json '
    '--out-csv reports/revision/tables/table_pipeline_storage_audit.csv',
    log_path='reports/revision/logs/final_pipeline_storage_audit_post_publish_strict_full_sha.log',
)


## Convenience Drive Copies

Copy final reviewer tables to a short Drive path for manual download/opening. These are convenience copies; the canonical artifacts remain under `reports/revision/` and the verified mirror.

In [ ]:
import json
import shutil
from datetime import datetime, timezone
from scripts.revision.common import sha256_file

FINAL_TABLE_EXPORT_DIR = DRIVE_ROOT / 'final_evidence_tables'
FINAL_TABLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_sources = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
optional_sources = [
    Path('reports/revision/manuscript/main_revised.pdf'),
    Path('reports/revision/manuscript/main_revised.pdf.txt'),
    Path('reports/revision/metrics/pipeline_storage_audit.json'),
    Path('reports/revision/tables/table_pipeline_storage_audit.csv'),
    Path('reports/revision/metrics/submission_forbidden_claim_scan.json'),
    Path('reports/revision/tables/table_submission_forbidden_claim_scan.csv'),
    Path('reports/revision/manifests/notebook_code_authority.json'),
    Path('reports/revision/metrics/submission_package_readiness.json'),
    Path('reports/revision/tables/table_submission_package_readiness.csv'),
    Path('reports/revision/manuscript/main_marked.tex'),
    Path('reports/revision/manuscript/main_marked.pdf'),
    Path('reports/revision/manifests/ptbxl_adaptation_analysis_lock_source_attestation.json'),
    Path('reports/revision/manifests/ptbxl_adaptation_analysis_lock.json'),
    Path('reports/revision/metrics/in_domain_paired_contract_refresh.json'),
    Path('reports/revision/metrics/ptbxl_fold_protocol_audit.json'),
    Path('reports/revision/tables/table_ptbxl_fold_protocol_audit.csv'),
    Path('reports/revision/tables/table_ptbxl_unsupported_only_sensitivity.csv'),
    Path('reports/revision/manifests/ptbxl_fold_protocol_audit_manifest.json'),
    Path('reports/revision/tables/table_hypothesis_control_finding_claim_boundary.csv'),
    Path('reports/revision/tables/table_hypothesis_control_finding_claim_boundary.tex'),
    Path('reports/revision/metrics/hypothesis_control_claim_boundary.json'),
    Path('reports/revision/metrics/matched_oof_calibration_summary.json'),
    Path('reports/revision/metrics/matched_oof_calibration_bootstrap.json'),
    Path('reports/revision/tables/table_matched_oof_calibration.csv'),
    Path('reports/revision/tables/table_matched_oof_calibration_coefficients.csv'),
    Path('reports/revision/tables/table_matched_oof_calibration.tex'),
    Path('reports/revision/tables/table_paired_matched_oof_calibration.csv'),
    Path('reports/revision/figures/figure_matched_calibration_audit.png'),
    Path('reports/revision/metrics/structured_ablation_5fold_summary.json'),
    Path('reports/revision/tables/table_structured_ablation_5fold.csv'),
    Path('reports/revision/tables/table_structured_ablation_5fold.tex'),
    Path('reports/revision/tables/table_paired_structured_ablation_5fold.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_learning_curve.csv'),
    Path('reports/revision/figures/figure_true_fewshot_head_ptbxl_learning_curve.png'),
    Path('reports/revision/metrics/physiological_interval_probe_summary.json'),
    Path('reports/revision/tables/table_physiological_interval_probe.csv'),
    Path('reports/revision/tables/table_physiological_interval_probe_contrasts.csv'),
    Path('reports/revision/tables/table_physiological_interval_probe.tex'),
    Path('reports/revision/tables/table_paired_inference_audit.csv'),
    Path('reports/revision/audits/notebook_forensic_audit.md'),
    Path('reports/revision/tables/table_notebook_cell_audit.csv'),
    Path('reports/revision/tables/table_reviewer_traceability.csv'),
    Path('reports/revision/tables/table_statistical_oracle_check.csv'),
    Path('reports/revision/tables/table_comparator_contract.csv'),
    Path('reports/revision/tables/table_forensic_rerun_dependencies.csv'),
    Path('reports/revision/metrics/artifact_provenance_audit.json'),
    Path('reports/revision/metrics/reviewer_gap_closure_status.json'),
    Path('reports/revision/tables/table_reviewer_gap_closure_status.csv'),
    Path('reports/revision/manifests/reviewer_gap_closure_manifest.json'),
    Path('reports/revision/tables/table_external_zero_target_ci_compact.csv'),
    Path('reports/revision/tables/table_external_zero_target_ci_compact.tex'),
    Path('reports/revision/tables/table_pooling_cross_dataset_compact.csv'),
    Path('reports/revision/tables/table_pooling_cross_dataset_compact.tex'),
    Path('reports/revision/tables/table_morphology_learnability_compact.csv'),
    Path('reports/revision/tables/table_morphology_learnability_compact.tex'),
    Path('reports/revision/tables/table_robustness_six_stress_compact.csv'),
    Path('reports/revision/tables/table_robustness_six_stress_compact.tex'),
    Path('reports/revision/metrics/morphology_learnability_summary.json'),
    Path('reports/revision/metrics/paired_morphology_learnability_comparison.json'),
    Path('reports/revision/tables/table_paired_morphology_learnability.csv'),
    Path('reports/revision/manifests/paired_morphology_learnability_manifest.json'),
    Path('reports/revision/metrics/pooling_sensitivity_external.csv'),
    Path('reports/revision/metrics/pooling_q3_paired_bootstrap.json'),
    Path('reports/revision/manifests/pooling_sensitivity_external_manifest.json'),
    Path('reports/revision/metrics/claim_readiness_gates.json'),
    Path('reports/revision/tables/table_claim_readiness_gates.csv'),
    Path('reports/revision/manifests/claim_readiness_gates_manifest.json'),
    Path('reports/revision/metrics/external_protocol_gate_summary.csv'),
    Path('reports/revision/tables/table_external_ptbxl_metrics.csv'),
    Path('reports/revision/tables/table_external_georgia_metrics.csv'),
    Path('reports/revision/tables/table_external_cpsc2021_metrics.csv'),
    Path('reports/revision/metrics/external_comparator_paired_summary.json'),
    Path('reports/revision/tables/table_external_comparator_paired.csv'),
    Path('reports/revision/metrics/external_comparator_paired_bootstrap_samples.csv'),
    Path('reports/revision/manifests/external_comparator_paired_manifest.json'),
    Path('reports/revision/metrics/group_safe_score_calibration_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_group_safe_score_calibration_ptbxl.csv'),
    Path('reports/revision/metrics/group_safe_score_calibration_ptbxl_bootstrap.json'),
    Path('reports/revision/manifests/group_safe_score_calibration_ptbxl_splits.npz'),
    Path('reports/revision/tables/table_group_safe_score_calibration_ptbxl_coefficients.csv'),
    Path('reports/revision/manifests/group_safe_score_calibration_ptbxl_manifest.json'),
    Path('reports/revision/metrics/true_fewshot_head_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_paired.csv'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_primary.csv'),
    Path('reports/revision/metrics/true_fewshot_head_ptbxl_bootstrap.json'),
    Path('reports/revision/tables/table_true_fewshot_head_ptbxl_coefficients.csv'),
    Path('reports/revision/manifests/true_fewshot_head_ptbxl_splits.npz'),
    Path('reports/revision/manifests/true_fewshot_head_ptbxl_manifest.json'),
    Path('reports/revision/metrics/representation_evidence_status.json'),
    Path('reports/revision/metrics/representation_probe_summary.json'),
    Path('reports/revision/tables/table_representation_probe.csv'),
    Path('reports/revision/tables/table_representation_probe_by_fold.csv'),
    Path('reports/revision/tables/table_representation_cka.csv'),
    Path('reports/revision/figures/figure_representation_audit.png'),
    Path('reports/revision/manifests/representation_probe_manifest.json'),
    Path('reports/revision/metrics/robustness_multicomparator_summary.csv'),
    Path('reports/revision/metrics/robustness_multicomparator_pairwise.json'),
    Path('reports/revision/tables/table_robustness_multicomparator.csv'),
    Path('reports/revision/manifests/robustness_multicomparator_manifest.json'),
    Path('reports/revision/metrics/robustness_full_vs_resnet_comparison.json'),
    Path('reports/revision/metrics/robustness_full_vs_raw_mamba_comparison.json'),
    Path('reports/revision/metrics/robustness_full_vs_transformer_comparison.json'),
    Path('reports/revision/figures/figure_calibration_audit.png'),
    Path('reports/revision/tables/table_calibration_ci_compact.csv'),
    Path('reports/revision/tables/table_paired_baseline_ci_compact.csv'),
    Path('reports/revision/tables/table_pooling_sensitivity_compact.csv'),
    Path('reports/revision/metrics/pooling_sensitivity_external.csv'),
    Path('reports/revision/tables/table_pooling_sensitivity_across_datasets.csv'),
    Path('reports/revision/tables/table_fold_pca_provenance.csv'),
    Path('reports/revision/tables/table_training_configuration.csv'),
    Path('reports/revision/tables/table_morphology_transform_contract.csv'),
    Path('reports/revision/manifests/morphology_transform_identity_gate.json'),
    Path('reports/revision/manifests/reviewer_completion_input_contract.json'),
    Path('reports/revision/manifests/marked_manuscript_manifest.json'),
]
# Include every authenticated profile-scoped robustness package selected/discovered by the generator.
for pattern in [
    'reports/revision/metrics/robustness_multicomparator*_summary.csv',
    'reports/revision/metrics/robustness_multicomparator*_pairwise.json',
    'reports/revision/metrics/robustness_full_vs_*_*_comparison.json',
    'reports/revision/tables/table_robustness_multicomparator*.csv',
    'reports/revision/manifests/robustness_multicomparator*_manifest.json',
]:
    optional_sources.extend(sorted(Path('.').glob(pattern)))
optional_sources = list(dict.fromkeys(optional_sources))
selected_sources = final_sources + [path for path in optional_sources if path.exists() and path.stat().st_size > 0]
export_name_sources = {}
for source_path in selected_sources:
    export_name_sources.setdefault(source_path.name, []).append(source_path)
export_name_collisions = {
    name: paths for name, paths in export_name_sources.items() if len(paths) > 1
}
if export_name_collisions:
    details = '; '.join(
        f"{{name}}={{[str(path) for path in paths]}}"
        for name, paths in sorted(export_name_collisions.items())
    )
    raise RuntimeError('Flat final-evidence export has basename collisions: ' + details)
expected_export_dir = (DRIVE_ROOT / 'final_evidence_tables').resolve()
if FINAL_TABLE_EXPORT_DIR.resolve() != expected_export_dir:
    raise RuntimeError(f'Refusing to clean unexpected export directory: {FINAL_TABLE_EXPORT_DIR}')
for child in FINAL_TABLE_EXPORT_DIR.iterdir():
    if child.is_file() or child.is_symlink():
        child.unlink()
    elif child.is_dir():
        shutil.rmtree(child)

for src in selected_sources:
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, FINAL_TABLE_EXPORT_DIR / src.name)
readme_path = FINAL_TABLE_EXPORT_DIR / 'README.md'
readme_path.write_text(
    '# ECG-RAMBA final evidence tables\n\n'
    'These are a generated convenience snapshot. They are never used as pipeline inputs. '
    'Canonical provenance remains reports/revision and the verified Drive mirror.\n\n'
    '- Final matrix and safe wording are the source of truth for manuscript/rebuttal wording.\n'
    '- Legacy row-split score calibration is provenance only and is not claim-ready.\n'
    '- Group-safe score calibration is not model-weight adaptation.\n'
    '- True few-shot means new linear classifier heads on frozen encoders, not end-to-end fine-tuning.\n'
    '- PTB-XL/Georgia are separate mapped record-level tasks; CPSC2021 is a separate 10-second AF/AFL mapped-window task.\n'
    '- Representation artifacts are audit/limitation evidence, not proof of disentanglement.\n'
    '- Learned-comparator robustness is claim-ready only when the canonical six-stress, five-metric, n_boot=1000 gate is complete.\n'
    f"- Selected learned-comparator robustness profile: {(payload.get('learned_comparator_robustness_audit') or {}).get('selected_profile') or 'none'}.\n"
    '- A marked manuscript is complete only when its latexdiff/LaTeX manifest marks it editorial_ready.\n',
    encoding='utf-8',
)

mirror_manifest_path = MIRROR_REVISION_ROOT / 'manifests' / 'mirror_manifest.json'
if not mirror_manifest_path.exists() or mirror_manifest_path.stat().st_size == 0:
    raise FileNotFoundError(f'Canonical mirror manifest missing before final export: {mirror_manifest_path}')
mirror_manifest = json.loads(mirror_manifest_path.read_text(encoding='utf-8'))
mirror_rows = {row.get('relative_path'): row for row in mirror_manifest.get('artifacts', [])}
export_rows = []
for src in selected_sources:
    relative = src.relative_to(Path('reports/revision')).as_posix()
    source_sha = sha256_file(src)
    mirror_row = mirror_rows.get(relative)
    if not mirror_row or mirror_row.get('sha256') != source_sha:
        raise RuntimeError(f'Final export source is not current in canonical mirror: {relative}')
    destination = FINAL_TABLE_EXPORT_DIR / src.name
    destination_sha = sha256_file(destination)
    if destination_sha != source_sha:
        raise RuntimeError(f'Final export checksum mismatch: {destination}')
    export_rows.append({
        'source_relative_path': f'reports/revision/{relative}',
        'export_name': destination.name,
        'size_bytes': destination.stat().st_size,
        'sha256': destination_sha,
    })
export_rows.append({
    'source_relative_path': 'generated:README.md',
    'export_name': readme_path.name,
    'size_bytes': readme_path.stat().st_size,
    'sha256': sha256_file(readme_path),
})
export_manifest = {
    'status': 'complete_convenience_snapshot',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'canonical_is_authoritative': True,
    'pipeline_input_allowed': False,
    'canonical_mirror': str(MIRROR_REVISION_ROOT),
    'canonical_mirror_manifest': str(mirror_manifest_path),
    'canonical_mirror_manifest_sha256': sha256_file(mirror_manifest_path),
    'artifacts': export_rows,
}
export_manifest_path = FINAL_TABLE_EXPORT_DIR / 'final_evidence_export_manifest.json'
export_manifest_path.write_text(json.dumps(export_manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('Convenience export dir:', FINAL_TABLE_EXPORT_DIR)
print('Convenience export manifest:', export_manifest_path, 'artifacts=', len(export_rows))


## Output Summary

In [ ]:
expected = [
    'reports/revision/metrics/final_evidence_matrix.json',
    'reports/revision/tables/table_final_evidence_matrix.csv',
    'reports/revision/tables/table_final_safe_wording.csv',
    'reports/revision/manifests/final_evidence_matrix_manifest.json',
    'reports/revision/metrics/claim_readiness_gates.json',
    'reports/revision/metrics/reviewer_gap_closure_status.json',
    'reports/revision/tables/table_external_zero_target_ci_compact.csv',
    'reports/revision/tables/table_pooling_cross_dataset_compact.csv',
    'reports/revision/tables/table_morphology_learnability_compact.csv',
    'reports/revision/tables/table_robustness_six_stress_compact.csv',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_evidence_matrix.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_safe_wording.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'final_evidence_export_manifest.json'),
]
for item in expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
print('Claim guidance:')
for key, value in payload.get('claim_guidance', {}).items():
    print(f'- {key}: {value}')
print('Claim readiness:', payload.get('claim_readiness_gates', {}).get('rows', []))
print('Selected robustness profile:', (payload.get('learned_comparator_robustness_audit') or {}).get('selected_profile'))
print('Selected robustness tier   :', ((payload.get('learned_comparator_robustness_audit') or {}).get('selected') or {}).get('evidence_tier'))
print('Notebook 07 complete. Use table_final_evidence_matrix.csv and table_final_safe_wording.csv as source of truth.')
